In [1]:
%pip install pandas scikit-learn streamlit joblib


Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/9.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.1 MB ? eta -:--:--
   -- ------------------------------------- 0.5/9.1 MB 1.6 MB/s eta 0:00:06
   --- ------------------------------------ 0.8/9.1 MB 1.7 MB/s eta 0:00:05
   ----- ---------------------------------- 1.3/9.1 MB 1.9 MB/s eta 0:00:05
   --------- ------------------------------ 2.1/9.1 MB 2.3 MB/s eta 0:00:04
   ----------- ---------------------------- 2.6/9.1 MB 2.4 MB/s eta 0:00:03
   -------------- ------------------------- 3.4/9.1 MB 2.6 MB/s eta 0:00:03
   ------------------- -------------------- 4.5/9.1 MB 2.9 MB/s eta 0:00:02
   ------------------------ --------------- 5.5/9.1 MB 3.2 MB/s eta 0:00:02
   ----------------------------- ---------- 6.8/9.1 MB 3.5 MB/s eta 0:00:01
   ----------------------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [1]:
print('ss')

ss


In [28]:
import os
os.startfile('.')

In [6]:
print(df.columns.tolist())

['ID', 'Project Code', 'PQ #', 'PO / SO #', 'ASN/DN #', 'Country', 'Managed By', 'Fulfill Via', 'Vendor INCO Term', 'Shipment Mode', 'PQ First Sent to Client Date', 'PO Sent to Vendor Date', 'Scheduled Delivery Date', 'Delivered to Client Date', 'Delivery Recorded Date', 'Product Group', 'Sub Classification', 'Vendor', 'Item Description', 'Molecule/Test Type', 'Brand', 'Dosage', 'Dosage Form', 'Unit of Measure (Per Pack)', 'Line Item Quantity', 'Line Item Value', 'Pack Price', 'Unit Price', 'Manufacturing Site', 'First Line Designation', 'Weight (Kilograms)', 'Freight Cost (USD)', 'Line Item Insurance (USD)']


In [11]:
import pandas as pd

# 1. Load the data
df = pd.read_csv("SCMS_Delivery_History_Dataset.csv")

# 2. Fix the dates (As done previously)
# Specify format='%Y-%m-%d' to fix the UserWarning
df['Scheduled Delivery Date'] = pd.to_datetime(df['Scheduled Delivery Date'], format='%Y-%m-%d', errors='coerce')
df['Delivered to Client Date'] = pd.to_datetime(df['Delivered to Client Date'], format='%Y-%m-%d', errors='coerce')

# 3. Calculate delay and is_late
df['actual_time'] = (df['Delivered to Client Date'] - df['Scheduled Delivery Date']).dt.days
df['is_late'] = (df['actual_time'] > 0).astype(int)

# 4. CRITICAL: Clean the Weight Column
# Why: This column often has text. 'errors=coerce' turns text like "10 (Air)" into NaN.
df['Weight (Kilograms)'] = pd.to_numeric(df['Weight (Kilograms)'], errors='coerce')

# Fill missing weights with the median so we don't lose data
df['Weight (Kilograms)'] = df['Weight (Kilograms)'].fillna(df['Weight (Kilograms)'].median())

print(f"Dataset ready. Late shipments: {df['is_late'].sum()} / {len(df)}")
df.head(10)

Dataset ready. Late shipments: 0 / 10324


,ID,Project Code,PQ #,PO / SO #,ASN/DN #,Country,Managed By,Fulfill Via,Vendor INCO Term,Shipment Mode,...,Line Item Value,Pack Price,Unit Price,Manufacturing Site,First Line Designation,Weight (Kilograms),Freight Cost (USD),Line Item Insurance (USD),actual_time,is_late
0,1,100-CI-T01,Pre-PQ Process,SCMS-4,ASN-8,Côte d'Ivoire,PMO - US,Direct Drop,EXW,Air,...,551.00,29.00,0.97,Ranbaxy Fine Chemicals LTD,Yes,13.0,780.34,NaN,NaN,0
1,3,108-VN-T01,Pre-PQ Process,SCMS-13,ASN-85,Vietnam,PMO - US,Direct Drop,EXW,Air,...,6200.00,6.20,0.03,"Aurobindo Unit III, India",Yes,358.0,4521.5,NaN,NaN,0
2,4,100-CI-T01,Pre-PQ Process,SCMS-20,ASN-14,Côte d'Ivoire,PMO - US,Direct Drop,FCA,Air,...,40000.00,80.00,0.80,ABBVIE GmbH & Co.KG Wiesbaden,Yes,171.0,1653.78,NaN,NaN,0
3,15,108-VN-T01,Pre-PQ Process,SCMS-78,ASN-50,Vietnam,PMO - US,Direct Drop,EXW,Air,...,127360.80,3.99,0.07,"Ranbaxy, Paonta Shahib, India",Yes,1855.0,16007.06,NaN,NaN,0
4,16,108-VN-T01,Pre-PQ Process,SCMS-81,ASN-55,Vietnam,PMO - US,Direct Drop,EXW,Air,...,121600.00,3.20,0.05,"Aurobindo Unit III, India",Yes,7590.0,45450.08,NaN,NaN,0
5,23,112-NG-T01,Pre-PQ Process,SCMS-87,ASN-57,Nigeria,PMO - US,Direct Drop,EXW,Air,...,2225.60,5.35,0.02,"Aurobindo Unit III, India",Yes,504.0,5920.42,NaN,NaN,0
6,44,110-ZM-T01,Pre-PQ Process,SCMS-139,ASN-130,Zambia,PMO - US,Direct Drop,DDU,Air,...,4374.00,32.40,0.36,MSD South Granville Australia,Yes,328.0,Freight Included in Commodity Cost,NaN,NaN,0
7,45,109-TZ-T01,Pre-PQ Process,SCMS-140,ASN-94,Tanzania,PMO - US,Direct Drop,EXW,Air,...,60834.55,3.65,0.06,"Aurobindo Unit III, India",Yes,1478.0,6212.41,NaN,NaN,0
8,46,112-NG-T01,Pre-PQ Process,SCMS-156,ASN-93,Nigeria,PMO - US,Direct Drop,EXW,Air,...,532.35,1.95,0.03,"Aurobindo Unit III, India",No,1047.0,See ASN-93 (ID#:1281),NaN,NaN,0
9,47,110-ZM-T01,Pre-PQ Process,SCMS-165,ASN-199,Zambia,PMO - US,Direct Drop,CIP,Air,...,115080.00,41.10,0.34,ABBVIE (Abbott) St. P'burg USA,Yes,643.0,Freight Included in Commodity Cost,NaN,NaN,0


In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10324 entries, 0 to 10323
Data columns (total 35 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   ID                            10324 non-null  int64         
 1   Project Code                  10324 non-null  object        
 2   PQ #                          10324 non-null  object        
 3   PO / SO #                     10324 non-null  object        
 4   ASN/DN #                      10324 non-null  object        
 5   Country                       10324 non-null  object        
 6   Managed By                    10324 non-null  object        
 7   Fulfill Via                   10324 non-null  object        
 8   Vendor INCO Term              10324 non-null  object        
 9   Shipment Mode                 9964 non-null   object        
 10  PQ First Sent to Client Date  10324 non-null  object        
 11  PO Sent to Vendor Date      

In [15]:
import pandas as pd
import numpy as np
import logging
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import joblib
import matplotlib.pyplot as plt

# Configure logging for production readiness
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)


class LogisticsPredictor:
    """
    End-to-end pipeline for predicting shipment delays.
    """
    
    def __init__(self, data_path):
        self.data_path = data_path
        self.model = None
        self.label_encoders = {}
        self.feature_importance = None
        
    def load_and_prepare_data(self):
        """Load data and perform initial cleaning"""
        logger.info(f"Loading data from {self.data_path}")
        df = pd.read_csv(self.data_path)
        
        # Convert date columns (as you've already done)
        date_columns = ['Scheduled Delivery Date', 'Delivered to Client Date']
        for col in date_columns:
            df[col] = pd.to_datetime(df[col], errors='coerce')
        
        # Drop rows with missing delivery dates
        initial_rows = len(df)
        df = df.dropna(subset=date_columns)
        logger.info(f"Dropped {initial_rows - len(df)} rows with missing dates")
        
        # Clean Weight column
        df['Weight (Kilograms)'] = pd.to_numeric(
            df['Weight (Kilograms)'], 
            errors='coerce'
        )
        weight_median = df['Weight (Kilograms)'].median()
        df['Weight (Kilograms)'].fillna(weight_median, inplace=True)
        
        # Define target variable
        df['is_late'] = (
            (df['Delivered to Client Date'] - df['Scheduled Delivery Date']).dt.days > 0
        ).astype(int)
        
        logger.info(f"Late shipments: {df['is_late'].sum()} ({df['is_late'].mean()*100:.2f}%)")
        
        self.df = df
        return df
    
    def feature_engineering(self):
        """
        ANSWER TO QUESTION 1: Smart Categorical Encoding
        
        Strategy:
        - Shipment Mode: One-Hot Encoding (low cardinality, ~4-5 modes)
        - Country: Frequency Encoding (high cardinality, preserves business signal)
        
        Why? One-hot for Country would create 100+ columns causing:
        - Overfitting on rare countries
        - Computational expense
        - Loss of generalization
        
        Frequency encoding captures "busy routes" signal that matters for delays.
        """
        logger.info("Starting feature engineering...")
        
        df = self.df.copy()
        
        # Handle missing Shipment Mode (360 missing values)
        df['Shipment Mode'].fillna('Unknown', inplace=True)
        
        # ONE-HOT ENCODING for Shipment Mode (categorical, low cardinality)
        shipment_dummies = pd.get_dummies(
            df['Shipment Mode'], 
            prefix='mode',
            drop_first=True  # Avoid multicollinearity
        )
        
        # FREQUENCY ENCODING for Country (high cardinality)
        country_freq = df['Country'].value_counts(normalize=True).to_dict()
        df['country_frequency'] = df['Country'].map(country_freq)
        
        # Additional business-relevant features
        df['weight_kg'] = df['Weight (Kilograms)']
        df['line_item_value'] = df['Line Item Value']
        
        # Combine features
        feature_columns = (
            list(shipment_dummies.columns) + 
            ['country_frequency', 'weight_kg', 'line_item_value']
        )
        
        X = pd.concat([shipment_dummies, df[['country_frequency', 'weight_kg', 'line_item_value']]], axis=1)
        y = df['is_late']
        
        logger.info(f"Feature matrix shape: {X.shape}")
        logger.info(f"Features: {list(X.columns)}")
        
        self.X = X
        self.y = y
        self.feature_names = list(X.columns)
        
        return X, y
    
    def train_model(self, tune_hyperparameters=True):
        """
        ANSWER TO QUESTION 2: RandomForest Training with Smart Hyperparameters
        
        Dataset-specific tuning for 10K rows:
        - max_depth: Prevent overfitting on complex interactions
        - min_samples_split: Require statistical significance before splits
        - class_weight: Handle imbalanced data (11.5% late shipments)
        - n_estimators: Balance accuracy vs training time
        """
        logger.info("Splitting data (80/20 train/test split)...")
        
        X_train, X_test, y_train, y_test = train_test_split(
            self.X, self.y, 
            test_size=0.2, 
            random_state=42,
            stratify=self.y  # Preserve class distribution
        )
        
        if tune_hyperparameters:
            logger.info("Tuning hyperparameters with GridSearchCV...")
            
            # Parameter grid optimized for this dataset size
            param_grid = {
                'n_estimators': [100, 200],
                'max_depth': [10, 15, 20],  # Prevent overfitting
                'min_samples_split': [50, 100],  # Require statistical significance
                'min_samples_leaf': [20, 30],  # Smooth decision boundaries
                'class_weight': ['balanced']  # Handle 11.5% positive class
            }
            
            rf = RandomForestClassifier(random_state=42)
            
            grid_search = GridSearchCV(
                rf, 
                param_grid, 
                cv=5,
                scoring='roc_auc',  # Better for imbalanced data than accuracy
                n_jobs=-1,
                verbose=1
            )
            
            grid_search.fit(X_train, y_train)
            self.model = grid_search.best_estimator_
            
            logger.info(f"Best parameters: {grid_search.best_params_}")
            logger.info(f"Best CV ROC-AUC: {grid_search.best_score_:.4f}")
        
        else:
            logger.info("Training with default hyperparameters...")
            self.model = RandomForestClassifier(
                n_estimators=200,
                max_depth=15,
                min_samples_split=50,
                min_samples_leaf=20,
                class_weight='balanced',
                random_state=42
            )
            self.model.fit(X_train, y_train)
        
        # Evaluate model
        y_pred = self.model.predict(X_test)
        y_pred_proba = self.model.predict_proba(X_test)[:, 1]
        
        logger.info("\n" + "="*50)
        logger.info("MODEL PERFORMANCE METRICS")
        logger.info("="*50)
        logger.info(f"\nClassification Report:\n{classification_report(y_test, y_pred)}")
        logger.info(f"\nROC-AUC Score: {roc_auc_score(y_test, y_pred_proba):.4f}")
        logger.info(f"\nConfusion Matrix:\n{confusion_matrix(y_test, y_pred)}")
        
        # Feature importance for business interpretation
        self.feature_importance = pd.DataFrame({
            'feature': self.feature_names,
            'importance': self.model.feature_importances_
        }).sort_values('importance', ascending=False)
        
        logger.info(f"\nTop 5 Features:\n{self.feature_importance.head()}")
        
        # Store test data for dashboard
        self.X_test = X_test
        self.y_test = y_test
        self.y_pred = y_pred
        self.y_pred_proba = y_pred_proba
        
        return self.model
    
    def save_model(self, filepath='logistics_model.pkl'):
        """Save trained model for deployment"""
        joblib.dump(self.model, filepath)
        logger.info(f"Model saved to {filepath}")
    
    def generate_dashboard_metrics(self):
        """
        ANSWER TO QUESTION 3: Key Metrics for CMA CGM Management Dashboard
        
        These 3 metrics matter to global shipping managers:
        1. Late Delivery Rate by Route (Country) - Where are bottlenecks?
        2. Shipment Mode Risk Score - Which transport modes are unreliable?
        3. Weight-based Delay Probability - Does cargo size predict delays?
        """
        
        dashboard_data = {}
        
        # Metric 1: Late Delivery Rate by Country (Top 10 routes)
        country_late_rate = (
            self.df.groupby('Country')['is_late']
            .agg(['mean', 'count'])
            .reset_index()
            .rename(columns={'mean': 'late_rate', 'count': 'shipments'})
            .sort_values('late_rate', ascending=False)
            .head(10)
        )
        dashboard_data['country_risk'] = country_late_rate
        
        # Metric 2: Shipment Mode Performance
        mode_performance = (
            self.df.groupby('Shipment Mode')['is_late']
            .agg(['mean', 'count'])
            .reset_index()
            .rename(columns={'mean': 'late_rate', 'count': 'shipments'})
            .sort_values('late_rate', ascending=False)
        )
        dashboard_data['mode_risk'] = mode_performance
        
        # Metric 3: Weight Distribution Impact
        self.df['weight_category'] = pd.cut(
            self.df['Weight (Kilograms)'],
            bins=[0, 100, 500, 1000, 10000],
            labels=['Light', 'Medium', 'Heavy', 'Very Heavy']
        )
        weight_impact = (
            self.df.groupby('weight_category')['is_late']
            .agg(['mean', 'count'])
            .reset_index()
            .rename(columns={'mean': 'late_rate', 'count': 'shipments'})
        )
        dashboard_data['weight_impact'] = weight_impact
        
        logger.info("\n" + "="*50)
        logger.info("DASHBOARD METRICS GENERATED")
        logger.info("="*50)
        logger.info(f"\nTop 5 Riskiest Countries:\n{country_late_rate.head()}")
        logger.info(f"\nShipment Mode Performance:\n{mode_performance}")
        logger.info(f"\nWeight Impact:\n{weight_impact}")
        
        return dashboard_data




if __name__ == "__main__":
    # Usage example
    predictor = LogisticsPredictor('SCMS_Delivery_History_Dataset.csv')
    
    # Step 1: Load and clean data
    df = predictor.load_and_prepare_data()
    
    # Step 2: Feature engineering with smart encoding
    X, y = predictor.feature_engineering()
    
    # Step 3: Train model with hyperparameter tuning
    model = predictor.train_model(tune_hyperparameters=True)
    
    # Step 4: Save model for deployment
    predictor.save_model('logistics_predictor.pkl')
    
    # Step 5: Generate dashboard metrics
    dashboard_metrics = predictor.generate_dashboard_metrics()
    
    logger.info("\n🎉 Training pipeline complete! Ready for Streamlit dashboard.")

2026-01-17 18:17:58,128 - INFO - Loading data from SCMS_Delivery_History_Dataset.csv
C:\Users\hammo\tmp\ipykernel_22100\265245128.py:46: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col], errors='coerce')
C:\Users\hammo\tmp\ipykernel_22100\265245128.py:46: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col], errors='coerce')
2026-01-17 18:17:58,657 - INFO - Dropped 0 rows with missing dates
C:\Users\hammo\tmp\ipykernel_22100\265245128.py:59: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the 

Fitting 5 folds for each of 24 candidates, totalling 120 fits


2026-01-17 18:18:48,621 - INFO - Best parameters: {'class_weight': 'balanced', 'max_depth': 15, 'min_samples_leaf': 20, 'min_samples_split': 50, 'n_estimators': 200}
2026-01-17 18:18:48,622 - INFO - Best CV ROC-AUC: 0.7470
2026-01-17 18:18:48,760 - INFO - 
2026-01-17 18:18:48,762 - INFO - MODEL PERFORMANCE METRICS
2026-01-17 18:18:48,763 - INFO - ==================================================
2026-01-17 18:18:48,791 - INFO - 
Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.75      0.84      1828
           1       0.26      0.68      0.38       237

    accuracy                           0.74      2065
   macro avg       0.60      0.72      0.61      2065
weighted avg       0.87      0.74      0.78      2065

2026-01-17 18:18:48,815 - INFO - 
ROC-AUC Score: 0.7781
2026-01-17 18:18:48,828 - INFO - 
Confusion Matrix:
[[1367  461]
 [  75  162]]
2026-01-17 18:18:48,878 - INFO - 
Top 5 Features:
             feature  importanc